# Experiment 2: reliability, reasoning, model size, and agreement logits

This notebook enumerates four controlled variables: all $2^K$ answer patterns, reasoning off/on, all configured shared reliability values, and model size. The seeded question sets are identical across every condition and model. With positional control enabled, every scenario is also shown in both candidate orders. Each selected model receives $|R\_VALUES| \times |REASONING\_VALUES| \times NUM\_QUESTION\_SETS \times 2^K \times 2$ rows (1,600 under the current configuration). `ENABLE_MTP` is the single top-level switch for using native MTP as the continuous completion backend.

Each row uses one continuous assistant generation. Both reasoning conditions use the same final-answer contract and generate a terminal `ANSWER: {candidate}` line. The reasoning-off condition emits only that line, while the reasoning-on condition explains its reasoning first. The runner stores the unmodified assistant completion and exact generated token IDs, locates the generated answer boundary, and teacher-forces the exact generated sequence for answer-boundary logits and activation capture. Missing or malformed answers remain in the dataset as compliance observations; they are never repaired.


In [1]:
from __future__ import annotations

import csv
import gc
import hashlib
import html
import json
import logging
import math
import os
import re
import sys
import threading
from collections import Counter, defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from statistics import correlation, fmean, median, pstdev
from time import monotonic

# This must be set before importing third-party packages such as Transformers,
# which may import torchao while probing optional quantization backends.
os.environ.setdefault("TORCHAO_FORCE_SKIP_LOADING_SO_FILES", "1")
logging.getLogger("torchao").setLevel(logging.ERROR)

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib.lines import Line2D
from transformers import AutoProcessor

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO_ROOT))

from mats_experiments.noisy_channel_bayesian import (
    CaptureSpec,
    ExecutionConfig,
    MetricSpec,
    ModelConfig,
    NoisyChannelBayesianEnvironment,
    QwenRunner,
    RandomSubsetQuestion,
    SGLangMTPConfig,
    SystemPrompt,
    TokenizerBinding,
    TranscriptDataset,
    TranscriptDatasetGenerator,
    XVsYPosteriorProbe,
    answer_patterns,
    get_answer_surface_logits,
)


## Configuration

`MODEL_SELECTION` accepts one model ID or a list. `R_VALUES` is the shared-reliability sweep. The question seed and schedules are shared across models, reliability values, and the two reasoning settings. With `LOAD_RESULTS_FROM_ARTIFACTS=True`, the notebook loads the saved transcript dataset and completed inference rows without loading a model or using the GPU; set it to `False` to generate and execute the experiment. `MAX_COMPLETION_TOKENS` is a watchdog for a continuous generation; generation normally stops at the tokenizer or model assistant-end token. It is not an experimental reasoning parameter.


In [2]:
N = 8
K = 3
R_VALUES = ["1/100", "1/10", "3/10", "5/10", "7/10", "9/10", "99/100"]
NUM_QUESTION_SETS = 10
CONTROL_POSITIONAL_BIAS = True
REASONING_VALUES = (False, True)
ENABLE_MTP = True  # Applies to the complete, single-pass generation for every row.

SYSTEM_PROMPT_TEXT = (
    "You are a Bayesian reasoner. Follow the user's game rules exactly. "
    "Use plaintext only. Do not use Markdown, headings, bullets, tables, code blocks, "
    "HTML, or any other formatting."
)
MODEL_SELECTION: str | list[str] = ["Qwen/Qwen3.5-9B"]

X = 2
Y = 7
SUBSET_SIZE = 4
SEED = 20260902

MODEL_DTYPE = "auto"
DEVICE_MAP = None
LOCAL_FILES_ONLY = True
LOAD_RESULTS_FROM_ARTIFACTS = False
MAX_COMPLETION_TOKENS = 4096
CHECKPOINT_EVERY_BATCHES = 25
LOG_INTERVAL_SECONDS = 30
EXPERIMENT_ROOT = REPO_ROOT / "artifacts" / "noisy_channel_bayesian_experiment_2"

SGLANG_PYTHON = REPO_ROOT / ".venv-sglang" / "bin" / "python"
COMPLETION_MTP_BY_MODEL = {
    "Qwen/Qwen3.5-4B": SGLangMTPConfig(
        enabled=ENABLE_MTP, python_executable=SGLANG_PYTHON,
        speculative_num_steps=3, speculative_eagle_topk=1,
        speculative_num_draft_tokens=4, context_length=8192,
        cuda_graph_max_batch_size=48,
    ),
    "Qwen/Qwen3.5-9B": SGLangMTPConfig(
        enabled=ENABLE_MTP, python_executable=SGLANG_PYTHON,
        speculative_num_steps=3, speculative_eagle_topk=1,
        speculative_num_draft_tokens=4, context_length=8192,
        cuda_graph_max_batch_size=24,
    ),
}
DEFAULT_COMPLETION_MTP = SGLangMTPConfig(
    enabled=ENABLE_MTP, python_executable=SGLANG_PYTHON, context_length=8192,
)
RUN_ID = (
    "agreement_logits_reliability_sweep_generated_answer_line_v1_"
    + ("mtp" if ENABLE_MTP else "transformers")
)
BATCH_SIZES_BY_MODEL = {
    "Qwen/Qwen3.5-4B": {"completion": 40, "capture": 4, "score": 16},
    "Qwen/Qwen3.5-9B": {"completion": 24, "capture": 2, "score": 8},
}
DEFAULT_BATCH_SIZES = {"completion": 4, "capture": 1, "score": 4}


In [3]:
def log_progress(message: str) -> None:
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    print(f"[{timestamp}] {message}", flush=True)


def checkpoint_row_count(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        with path.open("r", encoding="utf-8") as handle:
            return sum(bool(line.strip()) for line in handle)
    except OSError:
        return 0


@contextmanager
def progress_heartbeat(label: str, checkpoint_path: Path | None = None):
    started = monotonic()
    stopped = threading.Event()

    def report() -> None:
        while not stopped.wait(LOG_INTERVAL_SECONDS):
            suffix = ""
            if checkpoint_path is not None:
                suffix = (
                    f"; completed rows={checkpoint_row_count(checkpoint_path)}/"
                    f"{EXPECTED_INFERENCE_ROWS_PER_MODEL}"
                )
            log_progress(
                f"{label}: still running; elapsed={(monotonic() - started) / 60:.1f} min"
                f"{suffix}"
            )

    log_progress(f"{label}: started")
    worker = threading.Thread(target=report, name=f"{label}-heartbeat", daemon=True)
    worker.start()
    succeeded = False
    try:
        yield
        succeeded = True
    finally:
        stopped.set()
        worker.join(timeout=1)
        status = "finished" if succeeded else "stopped with an error"
        log_progress(f"{label}: {status} after {(monotonic() - started) / 60:.1f} min")


def normalize_model_ids(selection: str | list[str]) -> list[str]:
    values = [selection] if isinstance(selection, str) else list(selection)
    if not values or not all(isinstance(value, str) and value.strip() for value in values):
        raise ValueError("MODEL_SELECTION must contain non-empty model IDs.")
    if len(values) != len(set(values)):
        raise ValueError("MODEL_SELECTION contains a duplicate model ID.")
    return values


def model_key(model_id: str) -> str:
    readable = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_id).strip("_")
    digest = hashlib.sha256(model_id.encode()).hexdigest()[:8]
    return f"{readable}_{digest}"


MODEL_IDS = normalize_model_ids(MODEL_SELECTION)
environments = tuple(
    NoisyChannelBayesianEnvironment(
        n=N, k=K, r_values=reliability,
        control_positional_bias=CONTROL_POSITIONAL_BIAS,
    )
    for reliability in R_VALUES
)
R_EXACT_VALUES = tuple(
    f"{environment.reliabilities[0].numerator}/{environment.reliabilities[0].denominator}"
    for environment in environments
)
R_FLOAT_VALUES = tuple(float(environment.reliabilities[0]) for environment in environments)
R_LABEL_BY_EXACT = dict(zip(R_EXACT_VALUES, R_VALUES, strict=True))
R_FLOAT_BY_EXACT = dict(zip(R_EXACT_VALUES, R_FLOAT_VALUES, strict=True))
R_INDEX_BY_EXACT = {value: index for index, value in enumerate(R_EXACT_VALUES)}
question = RandomSubsetQuestion(subset_size=SUBSET_SIZE, replacement=False, sort=True)
probes = tuple(
    XVsYPosteriorProbe(
        x=X, y=Y, reasoning=reasoning, allow_same=False,
        call_layout="conversation",
    )
    for reasoning in REASONING_VALUES
)
system_prompt = SystemPrompt(SYSTEM_PROMPT_TEXT)
metric_spec = MetricSpec(sequence_scores=False)
capture_spec = CaptureSpec(
    logits_boundaries=("answer",),
    logits_scope="answer_surfaces",
    #streams=("resid_post",),
    #layers="all",
    #tokens="all",
    every_decode_position=False,
)

assert tuple(probe.reasoning for probe in probes) == REASONING_VALUES
assert all((probe.x, probe.y, probe.allow_same) == (X, Y, False) for probe in probes)
assert all(config.enabled is ENABLE_MTP for config in COMPLETION_MTP_BY_MODEL.values())
assert DEFAULT_COMPLETION_MTP.enabled is ENABLE_MTP
assert len(environments) == len(R_VALUES) == len(set(R_EXACT_VALUES))
assert all(environment.shared_reliability for environment in environments)
assert CONTROL_POSITIONAL_BIAS
EXPECTED_INFERENCE_ROWS_PER_MODEL = (
    len(R_VALUES) * len(REASONING_VALUES) * NUM_QUESTION_SETS * 2**K * 2
)
print({
    "models": MODEL_IDS,
    "reasoning_values": REASONING_VALUES,
    "reliability_values": R_VALUES,
    "load_results_from_artifacts": LOAD_RESULTS_FROM_ARTIFACTS,
    "mtp_enabled": ENABLE_MTP,
    "evidence_scenarios_per_reliability_reasoning_value": NUM_QUESTION_SETS * 2**K,
    "inference_rows_per_model": EXPECTED_INFERENCE_ROWS_PER_MODEL,
    "artifact_root": str(EXPERIMENT_ROOT),
})


{'models': ['Qwen/Qwen3.5-9B'], 'reasoning_values': (False, True), 'reliability_values': ['1/100', '1/10', '3/10', '5/10', '7/10', '9/10', '99/100'], 'load_results_from_artifacts': False, 'mtp_enabled': True, 'evidence_scenarios_per_reliability_reasoning_value': 80, 'inference_rows_per_model': 2240, 'artifact_root': '/workspace/MATS/artifacts/noisy_channel_bayesian_experiment_2'}


## Agreement fields and paired prompt validation

For candidate $c$, agreement is 1 when the report equals the truthful YES/NO answer predicted by $c$. Every reliability and reasoning condition shares the same seeded questions, exhaustive reports, and candidate order. Within each reliability, the reasoning variants also share the reliability statement, answer rules, parameter substitutions, and final-answer format; only the instruction controlling whether reasoning precedes the final answer line differs.


In [4]:
def row_reliability_exact(row: dict[str, object]) -> str:
    values = tuple(str(value) for value in row["reliabilities_exact"])
    assert len(values) == K and len(set(values)) == 1
    return values[0]


def question_schedules(
    dataset: TranscriptDataset, reasoning: bool, reliability_exact: str
) -> list[list[list[int]]]:
    return [
        next(
            row for row in dataset
            if row["question_set_index"] == index
            and row["reasoning"] is reasoning
            and row_reliability_exact(row) == reliability_exact
        )["membership_sets"]
        for index in range(NUM_QUESTION_SETS)
    ]


def validate_exhaustive_dataset(dataset: TranscriptDataset) -> None:
    expected_patterns = [
        "".join("Y" if value == "YES" else "N" for value in pattern)
        for pattern in answer_patterns(K)
    ]
    assert len(dataset) == EXPECTED_INFERENCE_ROWS_PER_MODEL
    assert dataset.manifest["environment_parameter_count"] == len(R_VALUES)
    assert dataset.manifest["parameterization_count"] == (
        len(R_VALUES) * len(REASONING_VALUES)
    )
    assert dataset.manifest["probe_parameter_count"] == 2
    assert tuple(dataset.manifest["reasoning_values"]) == REASONING_VALUES
    assert dataset.manifest["num_question_sets"] == NUM_QUESTION_SETS
    assert dataset.manifest["presentations_per_scenario"] == 2
    assert all(row["n"] == N and row["k"] == K for row in dataset)
    assert all(row["candidate_1"] == X and row["candidate_2"] == Y for row in dataset)
    assert all(
        row["messages"][0] == {"role": "system", "content": SYSTEM_PROMPT_TEXT}
        for row in dataset
    )
    schedules = {}
    for environment_index, reliability_exact in enumerate(R_EXACT_VALUES):
        for probe_index, reasoning in enumerate(REASONING_VALUES):
            selected = [
                row for row in dataset
                if row["reasoning"] is reasoning
                and row_reliability_exact(row) == reliability_exact
            ]
            assert len(selected) == NUM_QUESTION_SETS * 2**K * 2
            assert all(
                row["environment_parameter_index"] == environment_index
                and row["probe_parameter_index"] == probe_index
                for row in selected
            )
            schedules[(reliability_exact, reasoning)] = question_schedules(
                dataset, reasoning, reliability_exact
            )
            for question_set_index in range(NUM_QUESTION_SETS):
                rows = [
                    row for row in selected
                    if row["question_set_index"] == question_set_index
                ]
                assert [row["answer_pattern"] for row in rows] == [
                    pattern for pattern in expected_patterns for _ in range(2)
                ]
    reference_schedule = schedules[(R_EXACT_VALUES[0], REASONING_VALUES[0])]
    assert all(schedule == reference_schedule for schedule in schedules.values())

    paired = defaultdict(dict)
    for row in dataset:
        key = (
            row_reliability_exact(row), row["question_set_index"],
            row["answer_pattern_index"],
            row["presentation_order"], tuple(row["candidate_value_order"]),
        )
        paired[key][bool(row["reasoning"])] = row
    assert len(paired) == len(R_VALUES) * NUM_QUESTION_SETS * 2**K * 2
    assert all(set(pair) == {False, True} for pair in paired.values())
    for pair in paired.values():
        off_row, on_row = pair[False], pair[True]
        fixed_fields = (
            "n", "k", "reliabilities_exact", "question_set_index",
            "answer_pattern_index", "answer_pattern", "questions",
            "membership_sets", "observed_reports", "candidate_1",
            "candidate_2", "presentation_index", "presentation_order",
            "candidate_value_order", "x", "y", "allow_same",
            "call_layout", "answer_prefix",
        )
        assert all(off_row[field] == on_row[field] for field in fixed_fields)
        permitted_line = (
            f"The permitted values are {off_row['x']}, {off_row['y']}."
        )
        off = off_row["messages"][-1]["content"]
        on = on_row["messages"][-1]["content"]
        off_prefix, off_separator, _ = off.partition(permitted_line)
        on_prefix, on_separator, _ = on.partition(permitted_line)
        assert off_separator == on_separator == permitted_line
        assert off_prefix == on_prefix
        tie_line = (
            f"If the two posterior probabilities are equal, output either "
            f"{off_row['x']} or {off_row['y']}; either value is valid."
        )
        assert tie_line in off and tie_line in on
        common_ending = "Do not output anything after the final answer line."
        assert off.endswith(common_ending) and on.endswith(common_ending)
        assert "Output only the final answer line." in off
        assert "Explain your reasoning before the final answer line." in on
        assert f"{off_row['answer_prefix']} {off_row['x']}" in off and on
        assert f"{off_row['answer_prefix']} {off_row['y']}" in off and on
        assert "at most" not in on
        assert " | " not in on + off

    matched_reliabilities = defaultdict(dict)
    for row in dataset:
        key = (
            bool(row["reasoning"]), row["question_set_index"],
            row["answer_pattern_index"], row["presentation_order"],
            tuple(row["candidate_value_order"]),
        )
        matched_reliabilities[key][row_reliability_exact(row)] = row
    assert len(matched_reliabilities) == (
        len(REASONING_VALUES) * NUM_QUESTION_SETS * 2**K * 2
    )
    assert all(set(group) == set(R_EXACT_VALUES) for group in matched_reliabilities.values())
    reliability_fixed_fields = (
        "n", "k", "question_set_index", "answer_pattern_index",
        "answer_pattern", "questions", "membership_sets",
        "observed_reports", "candidate_1", "candidate_2",
        "presentation_index", "presentation_order",
        "candidate_value_order", "reasoning",
        "agreement_candidate_1_by_question",
        "agreement_candidate_2_by_question",
    )
    for group in matched_reliabilities.values():
        reference = group[R_EXACT_VALUES[0]]
        assert all(
            row[field] == reference[field]
            for row in group.values() for field in reliability_fixed_fields
        )


def print_serialized_prompt_pair(
    dataset: TranscriptDataset, *, model_id: str
) -> None:
    reliability_exact = "9/10" if "9/10" in R_EXACT_VALUES else R_EXACT_VALUES[0]
    example_rows = [
        row for row in dataset
        if row["question_set_index"] == 0
        and row["answer_pattern_index"] == 0
        and row["presentation_order"] == "C1_C2"
        and tuple(row["candidate_value_order"]) == (X, Y)
        and row_reliability_exact(row) == reliability_exact
    ]
    by_reasoning = {bool(row["reasoning"]): row for row in example_rows}
    assert len(example_rows) == 2 and set(by_reasoning) == {False, True}
    off_row, on_row = by_reasoning[False], by_reasoning[True]
    fixed_fields = (
        "n", "k", "reliabilities_exact", "question_set_index",
        "answer_pattern_index", "answer_pattern", "questions",
        "membership_sets", "observed_reports", "candidate_1",
        "candidate_2", "presentation_index", "presentation_order",
        "candidate_value_order", "x", "y", "allow_same",
        "call_layout", "answer_prefix",
    )
    assert all(off_row[field] == on_row[field] for field in fixed_fields)

    print("\n" + "=" * 100)
    print(f"PAIRED SERIALIZED PROMPT EXAMPLE: {model_id}")
    print({
        "question_set_index": off_row["question_set_index"],
        "answer_pattern_index": off_row["answer_pattern_index"],
        "answer_pattern": off_row["answer_pattern"],
        "reliability": R_LABEL_BY_EXACT[reliability_exact],
        "presentation_order": off_row["presentation_order"],
        "candidate_value_order": off_row["candidate_value_order"],
        "membership_sets": off_row["membership_sets"],
        "observed_reports": off_row["observed_reports"],
    })
    for reasoning in REASONING_VALUES:
        print(f"\n--- reasoning={reasoning} serialized_prompt ---")
        print(by_reasoning[reasoning]["serialized_prompt"])
    print("=" * 100)


## Load saved results or generate once, then probe the actual answer boundary

When `LOAD_RESULTS_FROM_ARTIFACTS=True`, this section uses `TranscriptDataset.load` for the saved prompts and reads the completed run's `results.jsonl` and `run_manifest.json`; it does not instantiate `AutoProcessor` or `QwenRunner`. Otherwise, the reasoning-on completion is left untouched and the runner finds the last `ANSWER:` marker and records its exact generated-token boundary. With reasoning off, `ANSWER:` is the end of the user message and the answer boundary includes the assistant-role tokens added by the chat template. A teacher-forced forward pass uses the exact prompt IDs plus the exact emitted sequence IDs (including an emitted stop token when available) to capture residual-stream activations and candidate logits. Rows with no usable boundary retain their completion and compliance failure but have missing answer-boundary logits.


In [5]:
datasets_by_model: dict[str, TranscriptDataset] = {}
results_by_model: dict[str, list[dict[str, object]]] = {}
balanced_results_by_model: dict[str, list[dict[str, object]]] = {}
run_dirs_by_model: dict[str, Path] = {}


def optional_mean(values: list[float | None]) -> float | None:
    return fmean(value for value in values if value is not None) if all(
        value is not None for value in values
    ) else None


def balance_positional_pairs(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    pairs = defaultdict(list)
    for row in rows:
        pairs[row["positional_control_pair_id"]].append(row)
    balanced_rows = []
    invariants = (
        "reasoning", "environment_parameter_index", "probe_parameter_index",
        "reliabilities_exact", "reliabilities", "question_set_index",
        "answer_pattern_index", "answer_pattern", "membership_sets",
        "observed_reports", "total_agreement_candidate_1",
        "total_agreement_candidate_2", "agreement_candidate_1_by_question",
        "agreement_candidate_2_by_question",
    )
    for pair_id, pair in pairs.items():
        by_order = {row["presentation_order"]: row for row in pair}
        assert set(by_order) == {"C1_C2", "C2_C1"}
        first, second = by_order["C1_C2"], by_order["C2_C1"]
        for field in invariants:
            assert first[field] == second[field], (pair_id, field)
        c1_values = [first["candidate_1_raw_logit"], second["candidate_1_raw_logit"]]
        c2_values = [first["candidate_2_raw_logit"], second["candidate_2_raw_logit"]]
        difference_values = [
            first["candidate_1_minus_candidate_2_raw_logit"],
            second["candidate_1_minus_candidate_2_raw_logit"],
        ]
        difference_first, difference_second = difference_values
        balanced = dict(first)
        balanced.update({
            "balanced_across_presentation_order": True,
            "candidate_1_raw_logit_order_c1_c2": c1_values[0],
            "candidate_1_raw_logit_order_c2_c1": c1_values[1],
            "candidate_2_raw_logit_order_c1_c2": c2_values[0],
            "candidate_2_raw_logit_order_c2_c1": c2_values[1],
            "candidate_1_raw_logit": optional_mean(c1_values),
            "candidate_2_raw_logit": optional_mean(c2_values),
            "candidate_1_minus_candidate_2_raw_logit": optional_mean(difference_values),
            "position_effect_on_candidate_difference": (
                0.5 * (float(difference_first) - float(difference_second))
                if difference_first is not None and difference_second is not None else None
            ),
            "pair_has_answer_boundary_logits": all(
                value is not None for value in difference_values
            ),
            "strict_answer_compliance_by_order": {
                order: bool(row["strict_answer_compliance"])
                for order, row in by_order.items()
            },
            "reasoning_length_tokens_by_order": {
                order: row["reasoning_length_tokens"] for order, row in by_order.items()
            },
            "completion_by_order": {
                order: row["full_completion"] for order, row in by_order.items()
            },
            "generated_token_ids_by_order": {
                order: row["generated_token_ids"] for order, row in by_order.items()
            },
            "presentation_artifacts": {
                order: {
                    key: row.get(key)
                    for key in ("row_id", "logit_path", "activation_path")
                }
                for order, row in by_order.items()
            },
        })
        balanced_rows.append(balanced)
    balanced_rows.sort(key=lambda row: (
        R_INDEX_BY_EXACT[row_reliability_exact(row)], bool(row["reasoning"]),
        int(row["question_set_index"]),
        int(row["answer_pattern_index"]),
    ))
    assert len(balanced_rows) == (
        len(R_VALUES) * len(REASONING_VALUES) * NUM_QUESTION_SETS * 2**K
    )
    return balanced_rows


def load_completed_run(
    experiment_dir: Path, *, run_id: str, model_id: str
) -> tuple[TranscriptDataset, TranscriptDataset, Path]:
    dataset = TranscriptDataset.load(experiment_dir)
    run_dir = experiment_dir / "runs" / run_id
    manifest_path = run_dir / "run_manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"No completed run manifest at {manifest_path}. "
            "Set LOAD_RESULTS_FROM_ARTIFACTS=False to run inference."
        )
    run_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if run_manifest.get("run_id") != run_id:
        raise ValueError(f"Run manifest ID does not match {run_id!r}.")
    stored_model = run_manifest.get("model", {})
    if stored_model.get("model_name_or_path") != model_id:
        raise ValueError(f"Saved run does not belong to model {model_id!r}.")
    results_path = run_dir / str(run_manifest.get("results_file", "results.jsonl"))
    result_rows = [
        json.loads(line)
        for line in results_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if run_manifest.get("row_count") != len(result_rows):
        raise ValueError("Run manifest row count does not match results.jsonl.")
    dataset_ids = [str(row["row_id"]) for row in dataset]
    result_ids = [str(row["row_id"]) for row in result_rows]
    if result_ids != dataset_ids:
        raise ValueError(
            "Saved results do not exactly match the configured transcript dataset."
        )
    results = TranscriptDataset(
        result_rows, manifest=run_manifest, experiment_dir=experiment_dir
    )
    return dataset, results, run_dir


reference_schedules = None
for model_index, model_id in enumerate(MODEL_IDS, start=1):
    sizes = BATCH_SIZES_BY_MODEL.get(model_id, DEFAULT_BATCH_SIZES)
    experiment_dir = EXPERIMENT_ROOT / model_key(model_id)
    if LOAD_RESULTS_FROM_ARTIFACTS:
        log_progress(
            f"Model {model_index}/{len(MODEL_IDS)} {model_id}: loading saved results"
        )
        dataset, results, run_dir = load_completed_run(
            experiment_dir, run_id=RUN_ID, model_id=model_id
        )
        runner = None
    else:
        log_progress(
            f"Model {model_index}/{len(MODEL_IDS)} {model_id}: loading tokenizer"
        )
        processor = AutoProcessor.from_pretrained(
            model_id, trust_remote_code=False, local_files_only=LOCAL_FILES_ONLY,
        )
        tokenizer = getattr(processor, "tokenizer", processor)
        binding = TokenizerBinding(tokenizer, enable_thinking=False)
        dataset = TranscriptDatasetGenerator(
            environment=environments,
            question=question,
            probe=probes,
            tokenizer_binding=binding,
            system_prompt=system_prompt,
            seed=SEED,
        ).generate(num_question_sets=NUM_QUESTION_SETS)
        dataset.save(experiment_dir)
        execution = ExecutionConfig(
            experiment_dir=experiment_dir,
            run_id=RUN_ID,
            batch_size=sizes["completion"],
            completion_batch_size=sizes["completion"],
            capture_batch_size=sizes["capture"],
            score_batch_size=sizes["score"],
            checkpoint_every_batches=CHECKPOINT_EVERY_BATCHES,
            max_completion_tokens=MAX_COMPLETION_TOKENS,
            resume=True,
            metrics=metric_spec,
            capture=capture_spec,
            completion_mtp=COMPLETION_MTP_BY_MODEL.get(
                model_id, DEFAULT_COMPLETION_MTP
            ),
        )
        runner = QwenRunner(ModelConfig(
            model_name_or_path=model_id, dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
            local_files_only=LOCAL_FILES_ONLY,
        ))
        results_path = experiment_dir / "runs" / execution.run_id / "results.jsonl"
        with progress_heartbeat(f"{model_id} inference", results_path):
            results = dataset.execute(runner, execution)
        run_dir = experiment_dir / "runs" / execution.run_id

    validate_exhaustive_dataset(dataset)
    print_serialized_prompt_pair(dataset, model_id=model_id)
    schedules = question_schedules(dataset, False, R_EXACT_VALUES[0])
    if reference_schedules is None:
        reference_schedules = schedules
    else:
        assert schedules == reference_schedules
    enriched = []
    for result in results:
        generated_ids = [int(value) for value in result["generated_token_ids"]]
        if runner is not None:
            assert result["completion"]["text"] == runner.tokenizer.decode(
                generated_ids, skip_special_tokens=True
            )
            assert result["candidate_1_answer_token_ids"] == runner.tokenizer.encode(
                str(result["candidate_1"]), add_special_tokens=False
            )
            assert result["candidate_2_answer_token_ids"] == runner.tokenizer.encode(
                str(result["candidate_2"]), add_special_tokens=False
            )
        assert result["teacher_forced_input_ids"][:result["teacher_forced_completion_start"]] == result["generation_input_ids"]
        assert result["generation_messages"][-1]["role"] == "user"
        assert result["generation_messages"][-1]["content"].endswith(
            "Do not output anything after the final answer line."
        )
        assert result["answer_boundary_source"] == "generated_completion"
        surface_logits = None
        if "answer_surface_raw_logits" in result:
            surface_logits = get_answer_surface_logits(result, run_dir, boundary="answer")
        row = dict(result)
        row["x_raw_logit"] = (
            surface_logits[str(result["x"])] if surface_logits is not None else None
        )
        row["y_raw_logit"] = (
            surface_logits[str(result["y"])] if surface_logits is not None else None
        )
        row["candidate_1_raw_logit"] = (
            surface_logits[str(result["candidate_1"])] if surface_logits is not None else None
        )
        row["candidate_2_raw_logit"] = (
            surface_logits[str(result["candidate_2"])] if surface_logits is not None else None
        )
        row["candidate_1_minus_candidate_2_raw_logit"] = (
            float(row["candidate_1_raw_logit"]) - float(row["candidate_2_raw_logit"])
            if surface_logits is not None else None
        )
        enriched.append(row)
    datasets_by_model[model_id] = dataset
    results_by_model[model_id] = enriched
    balanced_results_by_model[model_id] = balance_positional_pairs(enriched)
    run_dirs_by_model[model_id] = run_dir
    if runner is not None:
        del runner, processor, tokenizer
    gc.collect()

assert results_by_model, "Load saved results or run inference before analysis."


[2026-09-04 06:52:08 UTC] Model 1/1 Qwen/Qwen3.5-9B: loading tokenizer
[2026-09-04 06:52:13 UTC] Qwen/Qwen3.5-9B inference: started
[2026-09-04 06:52:43 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=0.5 min; completed rows=2240/2240
SGLang native MTP ready for Qwen/Qwen3.5-9B after 32.0s (NEXTN steps=3, topk=1, draft_tokens=4).
[2026-09-04 06:53:13 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=1.0 min; completed rows=2240/2240
[2026-09-04 06:53:43 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=1.5 min; completed rows=2240/2240
[2026-09-04 06:54:13 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=2.0 min; completed rows=2240/2240
[2026-09-04 06:54:43 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=2.5 min; completed rows=2240/2240
[2026-09-04 06:55:13 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=3.0 min; completed rows=2240/2240
[2026-09-04 06:55:43 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=3.5 min; completed rows=2240/2240
[

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

[2026-09-04 06:57:04 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=4.9 min; completed rows=2240/2240
[2026-09-04 06:57:48 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=5.6 min; completed rows=2240/2240
[2026-09-04 06:58:18 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=6.1 min; completed rows=2240/2240
[2026-09-04 06:58:48 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=6.6 min; completed rows=2240/2240
[2026-09-04 06:59:18 UTC] Qwen/Qwen3.5-9B inference: still running; elapsed=7.1 min; completed rows=2240/2240
[2026-09-04 06:59:18 UTC] Qwen/Qwen3.5-9B inference: finished after 7.1 min

PAIRED SERIALIZED PROMPT EXAMPLE: Qwen/Qwen3.5-9B
{'question_set_index': 0, 'answer_pattern_index': 0, 'answer_pattern': 'YYY', 'reliability': '9/10', 'presentation_order': 'C1_C2', 'candidate_value_order': [2, 7], 'membership_sets': [[3, 5, 6, 8], [1, 5, 7, 8], [1, 4, 5, 6]], 'observed_reports': ['YES', 'YES', 'YES']}

--- reasoning=False serialized_prompt ---
<|im_star

## Analysis helpers

All logit correlations use order-balanced rows that have an observed terminal-candidate boundary in both presentation orders. Semantic compliance accepts whitespace around the terminal candidate; exact-format compliance records whether the requested `ANSWER:<candidate>` form was followed literally. Both summaries use every raw completion, including missing-marker and missing-boundary rows.


In [6]:
RELIABILITY_MARKERS = ("o", "s", "^", "D", "v", "<", ">", "p", "h", "H", "d")
RELIABILITY_COLORS = (
    "tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple",
    "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan", "black",
)
assert len(R_EXACT_VALUES) <= len(RELIABILITY_MARKERS) == len(RELIABILITY_COLORS)
RELIABILITY_COLOR_BY_EXACT = {
    reliability_exact: RELIABILITY_COLORS[index]
    for index, reliability_exact in enumerate(R_EXACT_VALUES)
}
RELIABILITY_MARKER_BY_EXACT = {
    reliability_exact: RELIABILITY_MARKERS[index]
    for index, reliability_exact in enumerate(R_EXACT_VALUES)
}


def reasoning_label(value: bool) -> str:
    return "On" if value else "Off"


def rows_at_reasoning(rows: list[dict[str, object]], reasoning: bool) -> list[dict[str, object]]:
    selected = [row for row in rows if bool(row["reasoning"]) is reasoning]
    assert selected, f"No rows found for reasoning={reasoning}."
    return selected


def rows_at_reliability(
    rows: list[dict[str, object]], reliability_exact: str
) -> list[dict[str, object]]:
    selected = [
        row for row in rows if row_reliability_exact(row) == reliability_exact
    ]
    assert selected, f"No rows found for reliability={reliability_exact}."
    return selected


def rows_at_condition(
    rows: list[dict[str, object]], *, reasoning: bool, reliability_exact: str
) -> list[dict[str, object]]:
    return rows_at_reliability(rows_at_reasoning(rows, reasoning), reliability_exact)


def reliability_color(reliability_exact: str) -> str:
    return RELIABILITY_COLOR_BY_EXACT[reliability_exact]


def reliability_marker(reliability_exact: str) -> str:
    return RELIABILITY_MARKER_BY_EXACT[reliability_exact]


def eligible_logit_rows(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    return [
        row for row in rows
        if row.get("candidate_1_raw_logit") is not None
        and row.get("candidate_2_raw_logit") is not None
    ]


def tied_ranks(values: list[float]) -> list[float]:
    order = sorted(range(len(values)), key=values.__getitem__)
    ranks = [0.0] * len(values)
    start = 0
    while start < len(order):
        end = start + 1
        while end < len(order) and values[order[end]] == values[order[start]]:
            end += 1
        rank = (start + 1 + end) / 2
        for position in range(start, end):
            ranks[order[position]] = rank
        start = end
    return ranks


def safe_correlation(left: list[float], right: list[float]) -> float:
    if len(left) < 2 or len(set(left)) < 2 or len(set(right)) < 2:
        return float("nan")
    return correlation(left, right)


def correlation_summary(rows: list[dict[str, object]]) -> dict[str, float | int]:
    selected = eligible_logit_rows(rows)
    agreement_difference = [
        float(row["total_agreement_candidate_1"]) - float(row["total_agreement_candidate_2"])
        for row in selected
    ]
    logit_difference = [
        float(row["candidate_1_minus_candidate_2_raw_logit"]) for row in selected
    ]
    return {
        "n": len(selected),
        "spearman": safe_correlation(tied_ranks(agreement_difference), tied_ranks(logit_difference)),
        "pearson": safe_correlation(agreement_difference, logit_difference),
    }


def reasoning_selector(description: str = "Reasoning") -> widgets.ToggleButtons:
    return widgets.ToggleButtons(
        options=[("Off", False), ("On", True)], value=False,
        description=description, style={"description_width": "90px"},
    )


def reliability_selector(description: str = "Reliability") -> widgets.Dropdown:
    return widgets.Dropdown(
        options=[(label, exact) for exact, label in zip(R_EXACT_VALUES, R_VALUES)],
        value=R_EXACT_VALUES[0], description=description,
        style={"description_width": "90px"},
    )


## Correlations and compliance at a selected reasoning value

Correlation metrics are computed separately for every fixed model, reasoning setting, and reliability. Compliance remains pooled over reliability values, so reliability is not added as an axis to the compliance summary.


In [7]:
def display_aggregate_tables(reasoning: bool) -> None:
    for model_id, balanced in balanced_results_by_model.items():
        selected = rows_at_reasoning(balanced, reasoning)
        raw = rows_at_reasoning(results_by_model[model_id], reasoning)
        breaks = Counter(row["compliance_break"] or "compliant" for row in raw)
        print(f"{model_id} — reasoning {reasoning_label(reasoning)}")
        display({
            "correlations by fixed reliability": {
                R_LABEL_BY_EXACT[reliability_exact]: correlation_summary(
                    rows_at_reliability(selected, reliability_exact)
                )
                for reliability_exact in R_EXACT_VALUES
            },
            "balanced scenarios across reliability values": len(selected),
            "balanced scenarios with logits across reliability values": len(
                eligible_logit_rows(selected)
            ),
            "semantic completion compliance": fmean(
                float(row["semantic_answer_compliance"]) for row in raw
            ),
            "exact answer-format compliance": fmean(
                float(row["exact_answer_format_compliance"]) for row in raw
            ),
            "compliance outcomes": dict(breaks),
        })


aggregate_widget = reasoning_selector()
aggregate_output = widgets.interactive_output(
    display_aggregate_tables, {"reasoning": aggregate_widget}
)
display(widgets.VBox([aggregate_widget, aggregate_output]))


## Raw candidate logits against agreement difference

The first two panels plot the separate raw logits for $C_1$ and $C_2$ against $a(C_1)-a(C_2)$. The third panel plots their semantic contrast. Use the widgets to select one fixed reliability and one reasoning condition at a time; the selected reliability retains its categorical color and marker. The dashed contrast line shows the finite exact Bayesian log-odds for $0 < r < 1$; the deterministic endpoints have infinite ideal contrasts and are represented only by their measured points.


In [8]:
def plot_agreement_logits(reasoning: bool, reliability_exact: str) -> None:
    for model_id, all_rows in balanced_results_by_model.items():
        rows = eligible_logit_rows(rows_at_condition(
            all_rows, reasoning=reasoning, reliability_exact=reliability_exact
        ))
        if not rows:
            print(
                f"{model_id}: no answer-boundary logits for reasoning "
                f"{reasoning_label(reasoning)}, reliability "
                f"{R_LABEL_BY_EXACT[reliability_exact]}"
            )
            continue
        fig, axes = plt.subplots(1, 3, figsize=(20, 5.2), sharex=True)
        grid = list(range(-K, K + 1))
        reliability = R_FLOAT_BY_EXACT[reliability_exact]
        color = reliability_color(reliability_exact)
        marker = reliability_marker(reliability_exact)
        agreement_difference = [
            int(row["total_agreement_candidate_1"])
            - int(row["total_agreement_candidate_2"])
            for row in rows
        ]
        axes[0].scatter(
            agreement_difference,
            [float(row["candidate_1_raw_logit"]) for row in rows],
            alpha=0.55, color=color, marker=marker,
        )
        axes[1].scatter(
            agreement_difference,
            [float(row["candidate_2_raw_logit"]) for row in rows],
            alpha=0.55, color=color, marker=marker,
        )
        axes[2].scatter(
            agreement_difference,
            [float(row["candidate_1_minus_candidate_2_raw_logit"]) for row in rows],
            alpha=0.55, color=color, marker=marker,
        )
        if 0.0 < reliability < 1.0:
            exact_weight = math.log(reliability / (1.0 - reliability))
            axes[2].plot(
                grid, [value * exact_weight for value in grid],
                color=color, linestyle="--", linewidth=1.0, alpha=0.75,
            )
        axes[0].set(title=f"raw logit(C1={X})", ylabel="order-balanced raw logit")
        axes[1].set(title=f"raw logit(C2={Y})", ylabel="order-balanced raw logit")
        axes[2].set(
            title="raw logit(C1) - raw logit(C2) (dashed: exact)",
            ylabel="semantic logit contrast",
        )
        for axis in axes:
            axis.set(xlabel="agreement(C1) - agreement(C2)", xticks=grid)
            axis.grid(alpha=0.25)
        fig.suptitle(
            f"{model_id} — reasoning {reasoning_label(reasoning)} — "
            f"r={R_LABEL_BY_EXACT[reliability_exact]} — n={len(rows)}"
        )
        fig.subplots_adjust(top=0.86, wspace=0.25)
        plt.show()


plot_reasoning_widget = reasoning_selector()
plot_reliability_widget = reliability_selector()
plot_output = widgets.interactive_output(
    plot_agreement_logits,
    {"reasoning": plot_reasoning_widget, "reliability_exact": plot_reliability_widget},
)
display(widgets.VBox([
    widgets.HBox([plot_reasoning_widget, plot_reliability_widget]), plot_output
]))


## Across-question posterior calibration and agreement grid

The widgets below hold reliability and reasoning fixed while pooling every question set and all $2^K$ answer patterns. Every plotted row is already averaged over the `C1_C2` and `C2_C1` presentations, and each question set has a distinct categorical color in all three scatter panels. Because this run captured only the two answer-surface logits, the LLM probability is the softmax restricted to $\{C_1,C_2\}$ after positional logit averaging. The mathematical probability is normalized over the same pair: $q(C_i)=p(C_i\mid D)/(p(C_1\mid D)+p(C_2\mid D))$. Rows for which that denominator is zero or the posterior is undefined are excluded from the probability plot.

The log-odds plot includes only rows with strictly positive mathematical probabilities for both candidates; exact endpoint ratios that are zero, infinite, or undefined are counted but not coerced onto a finite axis. In the agreement grid, `n` sums the position-balanced answer-pattern rows from every question set in that agreement cell, while each displayed logit is their across-question mean.


In [15]:
QUESTION_SET_CMAP = plt.get_cmap("tab10", NUM_QUESTION_SETS)


def question_set_color(question_set_index: int) -> object:
    return QUESTION_SET_CMAP(question_set_index)


def question_set_legend_handles() -> list[Line2D]:
    return [
        Line2D(
            [], [], color=question_set_color(index), marker="o",
            linestyle="None", label=f"Question set {index}",
        )
        for index in range(NUM_QUESTION_SETS)
    ]


def stable_sigmoid(value: float) -> float:
    if value >= 0:
        return 1.0 / (1.0 + math.exp(-value))
    exponential = math.exp(value)
    return exponential / (1.0 + exponential)


def pairwise_mathematical_probabilities(
    row: dict[str, object],
) -> tuple[float, float] | None:
    candidate_1 = row.get("candidate_1_posterior")
    candidate_2 = row.get("candidate_2_posterior")
    if candidate_1 is None or candidate_2 is None:
        return None
    total = float(candidate_1) + float(candidate_2)
    if total <= 0.0:
        return None
    return float(candidate_1) / total, float(candidate_2) / total


def pairwise_llm_probabilities(
    row: dict[str, object],
) -> tuple[float, float] | None:
    contrast = row.get("candidate_1_minus_candidate_2_raw_logit")
    if contrast is None:
        return None
    candidate_1 = stable_sigmoid(float(contrast))
    return candidate_1, 1.0 - candidate_1


def association_metrics(
    x_values: list[float], y_values: list[float]
) -> dict[str, float | int]:
    return {
        "n": len(x_values),
        "pearson": safe_correlation(x_values, y_values),
        "spearman": safe_correlation(tied_ranks(x_values), tied_ranks(y_values)),
    }


def metric_text(value: float | int) -> str:
    numeric = float(value)
    return f"{numeric:.3f}" if math.isfinite(numeric) else "undefined"


def aggregate_question_rows(
    rows: list[dict[str, object]], *, reasoning: bool, reliability_exact: str,
) -> list[dict[str, object]]:
    selected = rows_at_condition(
        rows, reasoning=reasoning, reliability_exact=reliability_exact
    )
    selected.sort(key=lambda row: (
        int(row["question_set_index"]), int(row["answer_pattern_index"]),
    ))
    assert len(selected) == NUM_QUESTION_SETS * 2**K, (reasoning, reliability_exact)
    return selected


def plot_aggregate_question_diagnostics(
    reasoning: bool, reliability_exact: str
) -> None:
    reliability_label = R_LABEL_BY_EXACT[reliability_exact]
    condition_label = (
        f"all {NUM_QUESTION_SETS} question sets — "
        f"reasoning {reasoning_label(reasoning)} — r={reliability_label}"
    )
    for model_id, all_rows in balanced_results_by_model.items():
        rows = aggregate_question_rows(
            all_rows, reasoning=reasoning, reliability_exact=reliability_exact,
        )
        correlation_rows: list[dict[str, object]] = []

        probability_points: dict[str, list[tuple[int, str, float, float]]] = {
            "C1": [], "C2": [],
        }
        for row in rows:
            mathematical = pairwise_mathematical_probabilities(row)
            llm = pairwise_llm_probabilities(row)
            if mathematical is None or llm is None:
                continue
            question_index = int(row["question_set_index"])
            pattern = str(row["answer_pattern"])
            probability_points["C1"].append((
                question_index, pattern, mathematical[0], llm[0],
            ))
            probability_points["C2"].append((
                question_index, pattern, mathematical[1], llm[1],
            ))

        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4), sharex=True, sharey=True)
        for axis, candidate_key, candidate_value in zip(
            axes, ("C1", "C2"), (X, Y)
        ):
            points = probability_points[candidate_key]
            mathematical_values = [point[2] for point in points]
            llm_values = [point[3] for point in points]
            metrics = association_metrics(mathematical_values, llm_values)
            correlation_rows.append({
                "plot": "candidate probability", "series": candidate_key, **metrics,
            })
            for question_index in range(NUM_QUESTION_SETS):
                question_points = [
                    point for point in points if point[0] == question_index
                ]
                axis.scatter(
                    [point[2] for point in question_points],
                    [point[3] for point in question_points],
                    color=question_set_color(question_index), alpha=0.7,
                )
            axis.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
            axis.set(
                title=(
                    f"{candidate_key}={candidate_value} — n={metrics['n']} — "
                    f"Pearson={metric_text(metrics['pearson'])}, "
                    f"Spearman={metric_text(metrics['spearman'])}"
                ),
                xlabel="mathematical pairwise posterior probability",
                ylabel="position-balanced LLM answer probability",
                xlim=(-0.03, 1.03), ylim=(-0.03, 1.03),
            )
            axis.grid(alpha=0.25)
        fig.legend(
            handles=question_set_legend_handles(), title="Question set",
            loc="center right", bbox_to_anchor=(0.995, 0.5), fontsize=8,
        )
        fig.suptitle(f"{model_id} — candidate posterior calibration — {condition_label}")
        fig.subplots_adjust(top=0.84, right=0.84, wspace=0.18)
        plt.show()

        log_odds_points: list[tuple[int, str, float, float]] = []
        for row in rows:
            posterior_1 = row.get("candidate_1_posterior")
            posterior_2 = row.get("candidate_2_posterior")
            contrast = row.get("candidate_1_minus_candidate_2_raw_logit")
            if (
                posterior_1 is None or posterior_2 is None or contrast is None
                or float(posterior_1) <= 0.0 or float(posterior_2) <= 0.0
            ):
                continue
            log_odds_points.append((
                int(row["question_set_index"]),
                str(row["answer_pattern"]),
                math.log(float(posterior_1) / float(posterior_2)),
                float(contrast),
            ))
        mathematical_log_odds = [point[2] for point in log_odds_points]
        llm_logit_contrasts = [point[3] for point in log_odds_points]
        log_odds_metrics = association_metrics(
            mathematical_log_odds, llm_logit_contrasts
        )
        correlation_rows.append({
            "plot": "log odds", "series": "C1-C2", **log_odds_metrics,
        })
        fig, axis = plt.subplots(figsize=(8.5, 6.0))
        if log_odds_points:
            for question_index in range(NUM_QUESTION_SETS):
                question_points = [
                    point for point in log_odds_points if point[0] == question_index
                ]
                axis.scatter(
                    [point[2] for point in question_points],
                    [point[3] for point in question_points],
                    color=question_set_color(question_index), alpha=0.7,
                )
            bounds = mathematical_log_odds + llm_logit_contrasts
            lower, upper = min(bounds), max(bounds)
            if lower == upper:
                lower, upper = lower - 1.0, upper + 1.0
            axis.plot(
                [lower, upper], [lower, upper], color="black",
                linestyle="--", linewidth=1, label="calibrated y=x",
            )
            axis.legend(
                handles=question_set_legend_handles() + [
                    Line2D(
                        [], [], color="black", linestyle="--",
                        linewidth=1, label="calibrated y=x",
                    )
                ],
                title="Question set", fontsize=8,
                loc="center left", bbox_to_anchor=(1.02, 0.5),
            )
        else:
            axis.text(
                0.5, 0.5, "No finite mathematical candidate log-odds",
                ha="center", va="center", transform=axis.transAxes,
            )
        axis.set(
            title=(
                f"{model_id} — posterior log-odds calibration — {condition_label}\n"
                f"n={log_odds_metrics['n']}, excluded={len(rows) - len(log_odds_points)}, "
                f"Pearson={metric_text(log_odds_metrics['pearson'])}, "
                f"Spearman={metric_text(log_odds_metrics['spearman'])}"
            ),
            xlabel="mathematical log(p(C1 | D) / p(C2 | D))",
            ylabel="position-balanced logit(C1) - logit(C2)",
        )
        axis.grid(alpha=0.25)
        fig.subplots_adjust(right=0.78)
        plt.show()

        agreement_cells: dict[tuple[int, int], list[dict[str, object]]] = defaultdict(list)
        for row in rows:
            agreement_cells[(
                int(row["total_agreement_candidate_1"]),
                int(row["total_agreement_candidate_2"]),
            )].append(row)
        cell_summaries: dict[tuple[int, int], dict[str, float | int | None]] = {}
        for agreement_1 in range(K + 1):
            for agreement_2 in range(K + 1):
                cell_rows = agreement_cells[(agreement_1, agreement_2)]
                logit_1 = [
                    float(row["candidate_1_raw_logit"]) for row in cell_rows
                    if row.get("candidate_1_raw_logit") is not None
                ]
                logit_2 = [
                    float(row["candidate_2_raw_logit"]) for row in cell_rows
                    if row.get("candidate_2_raw_logit") is not None
                ]
                contrasts = [
                    float(row["candidate_1_minus_candidate_2_raw_logit"])
                    for row in cell_rows
                    if row.get("candidate_1_minus_candidate_2_raw_logit") is not None
                ]
                cell_summaries[(agreement_1, agreement_2)] = {
                    "n": len(cell_rows),
                    "logit_1": fmean(logit_1) if logit_1 else None,
                    "logit_2": fmean(logit_2) if logit_2 else None,
                    "contrast": fmean(contrasts) if contrasts else None,
                }
        contrast_grid = [
            [
                float(cell_summaries[(agreement_1, agreement_2)]["contrast"])
                if cell_summaries[(agreement_1, agreement_2)]["contrast"] is not None
                else float("nan")
                for agreement_2 in range(K + 1)
            ]
            for agreement_1 in range(K + 1)
        ]
        finite_contrasts = [
            float(summary["contrast"]) for summary in cell_summaries.values()
            if summary["contrast"] is not None
        ]
        limit = max((abs(value) for value in finite_contrasts), default=1.0) or 1.0
        fig, axis = plt.subplots(figsize=(9.0, 7.2))
        image = axis.imshow(
            contrast_grid, cmap="coolwarm", vmin=-limit, vmax=limit,
            origin="upper",
        )
        for agreement_1 in range(K + 1):
            for agreement_2 in range(K + 1):
                summary = cell_summaries[(agreement_1, agreement_2)]
                if summary["contrast"] is None:
                    text = f"n={summary['n']}\nz(C1)=—\nz(C2)=—\nΔz=—"
                    text_color = "black"
                else:
                    contrast = float(summary["contrast"])
                    text = (
                        f"n={summary['n']}\n"
                        f"z(C1)={float(summary['logit_1']):.2f}\n"
                        f"z(C2)={float(summary['logit_2']):.2f}\n"
                        f"Δz={contrast:.2f}"
                    )
                    text_color = "white" if abs(contrast) > 0.55 * limit else "black"
                axis.text(
                    agreement_2, agreement_1, text, ha="center", va="center",
                    fontsize=8, color=text_color,
                )
        axis.set(
            title=f"{model_id} — agreement-cell logits — {condition_label}",
            xlabel="agreement count for C2", ylabel="agreement count for C1",
            xticks=range(K + 1), yticks=range(K + 1),
        )
        axis.set_xticks([value - 0.5 for value in range(K + 2)], minor=True)
        axis.set_yticks([value - 0.5 for value in range(K + 2)], minor=True)
        axis.grid(which="minor", color="white", linewidth=1.5)
        axis.tick_params(which="minor", bottom=False, left=False)
        colorbar = fig.colorbar(image, ax=axis, pad=0.02)
        colorbar.set_label("mean position-balanced logit(C1) - logit(C2)")
        fig.subplots_adjust(top=0.88)
        plt.show()

        table_rows = []
        for summary in correlation_rows:
            table_rows.append(
                "<tr>"
                f"<td>{html.escape(str(summary['plot']))}</td>"
                f"<td>{html.escape(str(summary['series']))}</td>"
                f"<td>{summary['n']}</td>"
                f"<td>{metric_text(summary['pearson'])}</td>"
                f"<td>{metric_text(summary['spearman'])}</td>"
                "</tr>"
            )
        display(HTML(
            f"<b>{html.escape(model_id)} — correlation summary — "
            f"{html.escape(condition_label)}</b>"
            "<table><thead><tr><th>Plot</th><th>Series</th><th>n</th>"
            "<th>Pearson</th><th>Spearman</th></tr></thead><tbody>"
            + "".join(table_rows) + "</tbody></table>"
        ))


aggregate_reasoning_widget = reasoning_selector()
aggregate_reliability_widget = reliability_selector()
aggregate_question_output = widgets.interactive_output(
    plot_aggregate_question_diagnostics,
    {
        "reasoning": aggregate_reasoning_widget,
        "reliability_exact": aggregate_reliability_widget,
    },
)
display(widgets.VBox([
    widgets.HBox([aggregate_reasoning_widget, aggregate_reliability_widget]),
    aggregate_question_output,
]))


In [16]:
def candidate_membership_count(
    row: dict[str, object], candidate_index: int,
) -> int:
    candidate = row[f"candidate_{candidate_index}"]
    return sum(
        candidate in membership_set
        for membership_set in row["membership_sets"]
    )


def position_balanced_candidate_logit(
    row: dict[str, object], candidate_index: int,
) -> float | None:
    logits = [
        row.get(f"candidate_{candidate_index}_raw_logit_order_c1_c2"),
        row.get(f"candidate_{candidate_index}_raw_logit_order_c2_c1"),
    ]
    if any(value is None for value in logits):
        return None
    return fmean(float(value) for value in logits if value is not None)


def plot_candidate_membership_agreement_grid(
    reasoning: bool, reliability_exact: str,
) -> None:
    reliability_label = R_LABEL_BY_EXACT[reliability_exact]
    condition_label = (
        f"all {NUM_QUESTION_SETS} question sets — "
        f"reasoning {reasoning_label(reasoning)} — r={reliability_label}"
    )
    for model_id, all_rows in balanced_results_by_model.items():
        rows = aggregate_question_rows(
            all_rows, reasoning=reasoning, reliability_exact=reliability_exact,
        )
        # Fold original C1 and C2 into one focal-candidate view. Each value is
        # first averaged across C1_C2 and C2_C1 presentation orders.
        cell_logits: dict[tuple[int, int], list[float]] = defaultdict(list)
        for row in rows:
            for candidate_index in (1, 2):
                logit = position_balanced_candidate_logit(row, candidate_index)
                if logit is None:
                    continue
                membership_count = candidate_membership_count(row, candidate_index)
                agreement_count = int(
                    row[f"total_agreement_candidate_{candidate_index}"]
                )
                assert 0 <= membership_count <= K
                assert 0 <= agreement_count <= K
                cell_logits[(membership_count, agreement_count)].append(logit)

        mean_grid = [
            [
                fmean(cell_logits[(membership_count, agreement_count)])
                if cell_logits[(membership_count, agreement_count)]
                else float("nan")
                for agreement_count in range(K + 1)
            ]
            for membership_count in range(K + 1)
        ]
        finite_means = [
            value for row_values in mean_grid for value in row_values
            if math.isfinite(value)
        ]
        limit = max((abs(value) for value in finite_means), default=1.0) or 1.0

        fig, axis = plt.subplots(figsize=(8.2, 7.0))
        image = axis.imshow(
            mean_grid, cmap="coolwarm", vmin=-limit, vmax=limit,
            origin="upper",
        )
        for membership_count in range(K + 1):
            for agreement_count in range(K + 1):
                logits = cell_logits[(membership_count, agreement_count)]
                mean_logit = fmean(logits) if logits else None
                text = (
                    f"n={len(logits)}\nz(C1)={mean_logit:.2f}"
                    if mean_logit is not None else "n=0\nz(C1)=—"
                )
                text_color = (
                    "white"
                    if mean_logit is not None and abs(mean_logit) > 0.55 * limit
                    else "black"
                )
                axis.text(
                    agreement_count, membership_count, text,
                    ha="center", va="center", fontsize=9, color=text_color,
                )
        axis.set(
            title=(
                f"{model_id} — focal-candidate membership/agreement logits — "
                f"{condition_label}\n"
                "C1 and C2 pooled after position balancing"
            ),
            xlabel="agreement count for focal C1",
            ylabel="membership count for focal C1",
            xticks=range(K + 1), yticks=range(K + 1),
        )
        axis.set_xticks([value - 0.5 for value in range(K + 2)], minor=True)
        axis.set_yticks([value - 0.5 for value in range(K + 2)], minor=True)
        axis.grid(which="minor", color="white", linewidth=1.5)
        axis.tick_params(which="minor", bottom=False, left=False)
        colorbar = fig.colorbar(image, ax=axis, pad=0.02)
        colorbar.set_label("mean position-balanced z(C1)")
        fig.subplots_adjust(top=0.84)
        plt.show()


candidate_grid_reasoning_widget = reasoning_selector()
candidate_grid_reliability_widget = reliability_selector()
candidate_grid_output = widgets.interactive_output(
    plot_candidate_membership_agreement_grid,
    {
        "reasoning": candidate_grid_reasoning_widget,
        "reliability_exact": candidate_grid_reliability_widget,
    },
)
display(widgets.VBox([
    widgets.HBox([
        candidate_grid_reasoning_widget, candidate_grid_reliability_widget,
    ]),
    candidate_grid_output,
]))


## Correlation with and without reasoning

These panels compare the two matched reasoning conditions directly for every model at the fixed reliability selected by the widget. No correlation combines reliability values. Each point uses only order-balanced scenarios with an observed answer boundary in both orders; labels show the retained sample size.


In [17]:
correlation_rows_by_model = {
    model_id: {
        reliability_exact: {
            reasoning: correlation_summary(rows_at_condition(
                rows, reasoning=reasoning, reliability_exact=reliability_exact
            ))
            for reasoning in REASONING_VALUES
        }
        for reliability_exact in R_EXACT_VALUES
    }
    for model_id, rows in balanced_results_by_model.items()
}
def plot_reasoning_correlations(reliability_exact: str) -> None:
    color = reliability_color(reliability_exact)
    marker = reliability_marker(reliability_exact)
    for model_id, summaries_by_reliability in correlation_rows_by_model.items():
        summaries = summaries_by_reliability[reliability_exact]
        fig, (ax_spearman, ax_pearson) = plt.subplots(
            1, 2, figsize=(14, 5.2), sharex=True
        )
        x_values = [0, 1]
        ax_spearman.plot(
            x_values,
            [float(summaries[value]["spearman"]) for value in REASONING_VALUES],
            marker=marker, color=color,
        )
        ax_pearson.plot(
            x_values,
            [float(summaries[value]["pearson"]) for value in REASONING_VALUES],
            marker=marker, color=color,
        )
        for axis, key in ((ax_spearman, "spearman"), (ax_pearson, "pearson")):
            for x_value, reasoning in zip(x_values, REASONING_VALUES):
                summary = summaries[reasoning]
                metric = float(summary[key])
                if math.isfinite(metric):
                    axis.annotate(
                        f"n={summary['n']}",
                        (x_value, metric), xytext=(4, 4),
                        textcoords="offset points", fontsize=7, color=color,
                    )
        for axis, title, ylabel in (
            (ax_spearman, "Rank alignment", "Spearman correlation"),
            (ax_pearson, "Linear alignment", "Pearson correlation"),
        ):
            axis.axhline(0, color="black", linewidth=0.8, alpha=0.6)
            axis.set(
                xlabel="Reasoning", ylabel=ylabel, title=title,
                xticks=[0, 1], xticklabels=["Off", "On"], ylim=(-1.05, 1.05),
            )
            axis.grid(alpha=0.25)
        fig.suptitle(f"{model_id} — r={R_LABEL_BY_EXACT[reliability_exact]}")
        fig.subplots_adjust(top=0.86, wspace=0.25)
        plt.show()


correlation_reliability_widget = reliability_selector()
correlation_output = widgets.interactive_output(
    plot_reasoning_correlations,
    {"reliability_exact": correlation_reliability_widget},
)
display(widgets.VBox([correlation_reliability_widget, correlation_output]))
correlation_rows_by_model


{'Qwen/Qwen3.5-9B': {'1/100': {False: {'n': 80,
    'spearman': 0.6051085776312889,
    'pearson': 0.6285029729437285},
   True: {'n': 80,
    'spearman': -0.8671788357326098,
    'pearson': -0.8670680212895424}},
  '1/10': {False: {'n': 80,
    'spearman': 0.6342168316378038,
    'pearson': 0.6403540904822306},
   True: {'n': 80,
    'spearman': -0.8637803000659964,
    'pearson': -0.905011587283327}},
  '3/10': {False: {'n': 80,
    'spearman': 0.6284835092611805,
    'pearson': 0.6482484167656632},
   True: {'n': 80,
    'spearman': -0.8597835360248571,
    'pearson': -0.9018084403120445}},
  '1/2': {False: {'n': 80,
    'spearman': 0.6802728878603905,
    'pearson': 0.6888009024500894},
   True: {'n': 80,
    'spearman': 0.28153166441652766,
    'pearson': 0.07795064695900739}},
  '7/10': {False: {'n': 80,
    'spearman': 0.6463112054767703,
    'pearson': 0.6857997580905029},
   True: {'n': 80,
    'spearman': 0.915158833745117,
    'pearson': 0.9115184943972003}},
  '9/10': {Fals

## Reasoning length, compliance, and completion statistics

Reasoning length is the exact number of generated assistant tokens from the start of the assistant turn through the final colon in the last `ANSWER:` marker. It is `None` when the marker is missing or is not aligned to a token boundary. The compliance bar and length histogram retain their original summaries pooled across reliability. The widget filters only the length-versus-evidence scatter to one reliability; compliance is shown there by filled versus hollow points. The plots retain malformed completions and separate compliance from boundary availability.


In [18]:
def completion_statistics(rows: list[dict[str, object]]) -> dict[str, object]:
    lengths = [
        int(row["reasoning_length_tokens"])
        for row in rows if row["reasoning_length_tokens"] is not None
    ]
    generated_lengths = [len(row["generated_token_ids"]) for row in rows]
    return {
        "rows": len(rows),
        "semantic_compliance_rate": fmean(float(row["semantic_answer_compliance"]) for row in rows),
        "exact_format_compliance_rate": fmean(float(row["exact_answer_format_compliance"]) for row in rows),
        "answer_marker_rate": fmean(float(row["answer_marker_present"]) for row in rows),
        "boundary_available_rate": fmean(
            float(row["answer_boundary_generated_token_count"] is not None) for row in rows
        ),
        "reasoning_length_count": len(lengths),
        "reasoning_length_mean": fmean(lengths) if lengths else None,
        "reasoning_length_median": median(lengths) if lengths else None,
        "reasoning_length_sd": pstdev(lengths) if len(lengths) > 1 else 0.0 if lengths else None,
        "reasoning_length_min": min(lengths) if lengths else None,
        "reasoning_length_max": max(lengths) if lengths else None,
        "generated_length_mean": fmean(generated_lengths),
        "hit_completion_cap": sum(bool(row["completion"]["hit_token_cap"]) for row in rows),
        "answer_whitespace_forms": dict(Counter(
            f"{row['answer_leading_whitespace']!r} + candidate + {row['answer_trailing_whitespace']!r}"
            for row in rows if row["semantic_answer_compliance"]
        )),
        "emitted_candidate_token_ids": dict(Counter(
            str(row["answer_value_generated_token_ids"])
            for row in rows if row["semantic_answer_compliance"]
        )),
        "terminal_stop_token_ids": dict(Counter(
            str(row["completion"].get("terminal_stop_token_id")) for row in rows
        )),
        "compliance_breaks": dict(Counter(row["compliance_break"] or "compliant" for row in rows)),
    }


reasoning_statistics_by_model = {
    model_id: {
        reasoning: completion_statistics(rows_at_reasoning(all_rows, reasoning))
        for reasoning in REASONING_VALUES
    }
    for model_id, all_rows in results_by_model.items()
}


def plot_reasoning_diagnostics(reliability_exact: str) -> None:
    color = reliability_color(reliability_exact)
    marker = reliability_marker(reliability_exact)
    for model_id, all_rows in results_by_model.items():
        off_rows = rows_at_reasoning(all_rows, False)
        on_rows = rows_at_reasoning(all_rows, True)
        available = [row for row in on_rows if row["reasoning_length_tokens"] is not None]
        compliant_lengths = [
            int(row["reasoning_length_tokens"]) for row in available
            if row["strict_answer_compliance"]
        ]
        broken_lengths = [
            int(row["reasoning_length_tokens"]) for row in available
            if not row["strict_answer_compliance"]
        ]
        fig, axes = plt.subplots(1, 3, figsize=(20, 5.2))
        compliance_rates = [
            fmean(float(row["strict_answer_compliance"]) for row in selected)
            for selected in (off_rows, on_rows)
        ]
        axes[0].bar(["Off", "On"], compliance_rates, color=["tab:grey", "tab:green"])
        axes[0].set(title="Semantic answer compliance", ylabel="fraction compliant", ylim=(0, 1.05))
        bins = min(30, max(5, int(math.sqrt(max(1, len(available))))))
        if compliant_lengths:
            axes[1].hist(compliant_lengths, bins=bins, alpha=0.65, label="compliant")
        if broken_lengths:
            axes[1].hist(broken_lengths, bins=bins, alpha=0.65, label="noncompliant")
        axes[1].set(title="Reasoning length", xlabel="tokens immediately before candidate", ylabel="rows")
        axes[1].legend(fontsize=8)
        scatter_rows = [
            row for row in available
            if row_reliability_exact(row) == reliability_exact
        ]
        for compliant in (True, False):
            selected = [
                row for row in scatter_rows
                if bool(row["strict_answer_compliance"]) is compliant
            ]
            if not selected:
                continue
            axes[2].scatter(
                [
                    int(row["total_agreement_candidate_1"])
                    - int(row["total_agreement_candidate_2"])
                    for row in selected
                ],
                [int(row["reasoning_length_tokens"]) for row in selected],
                alpha=0.55, marker=marker, edgecolors=color,
                facecolors=color if compliant else "none", linewidths=0.9,
            )
        axes[2].set(
            title=f"Length versus evidence difference — r={R_LABEL_BY_EXACT[reliability_exact]}",
            xlabel="agreement(C1) - agreement(C2)", ylabel="reasoning length (tokens)",
        )
        axes[2].legend(
            handles=[
                Line2D([], [], color=color, marker=marker, linestyle="None", label="compliant"),
                Line2D(
                    [], [], color=color, marker=marker, markerfacecolor="none",
                    linestyle="None", label="noncompliant",
                ),
            ],
            title="answer format", fontsize=8,
        )
        for axis in axes:
            axis.grid(alpha=0.25)
        fig.suptitle(model_id)
        fig.subplots_adjust(top=0.86, wspace=0.25)
        plt.show()


reasoning_reliability_widget = reliability_selector()
reasoning_output = widgets.interactive_output(
    plot_reasoning_diagnostics,
    {"reliability_exact": reasoning_reliability_widget},
)
display(widgets.VBox([reasoning_reliability_widget, reasoning_output]))
reasoning_statistics_by_model


{'Qwen/Qwen3.5-9B': {False: {'rows': 1120,
   'semantic_compliance_rate': 1.0,
   'exact_format_compliance_rate': 1.0,
   'answer_marker_rate': 1.0,
   'boundary_available_rate': 1.0,
   'reasoning_length_count': 0,
   'reasoning_length_mean': None,
   'reasoning_length_median': None,
   'reasoning_length_sd': None,
   'reasoning_length_min': None,
   'reasoning_length_max': None,
   'generated_length_mean': 6.0,
   'hit_completion_cap': 0,
   'answer_whitespace_forms': {"' ' + candidate + ''": 1120},
   'emitted_candidate_token_ids': {'[22]': 921, '[17]': 199},
   'terminal_stop_token_ids': {'248046': 1120},
   'compliance_breaks': {'compliant': 1120}},
  True: {'rows': 1120,
   'semantic_compliance_rate': 1.0,
   'exact_format_compliance_rate': 1.0,
   'answer_marker_rate': 1.0,
   'boundary_available_rate': 1.0,
   'reasoning_length_count': 1120,
   'reasoning_length_mean': 684.3901785714286,
   'reasoning_length_median': 677.0,
   'reasoning_length_sd': 73.1908982190475,
   'reason

## Raw wide truth tables

One CSV is exported per model, reasoning value, and fixed reliability. Missing logits remain empty cells, so compliance failures are not silently dropped from the tabular record.


In [19]:
def format_optional(value: object) -> str:
    return "" if value is None else f"{float(value):.8f}"


def raw_truth_table(rows: list[dict[str, object]]) -> tuple[list[str], list[list[str]]]:
    by_key = {
        (int(row["answer_pattern_index"]), int(row["question_set_index"])): row
        for row in rows
    }
    headers = ["answer_pattern", *[f"report_i{i}" for i in range(1, K + 1)]]
    for j in range(1, NUM_QUESTION_SETS + 1):
        headers.extend([
            f"agreement_j{j}_c1", f"agreement_j{j}_c2",
            f"raw_logit_j{j}_c1", f"raw_logit_j{j}_c2",
            f"raw_logit_j{j}_c1_minus_c2", f"has_boundary_logits_j{j}",
        ])
    matrix = []
    for pattern_index, reports in enumerate(answer_patterns(K)):
        pattern = "".join("Y" if report == "YES" else "N" for report in reports)
        output = [pattern, *reports]
        for question_set_index in range(NUM_QUESTION_SETS):
            row = by_key[(pattern_index, question_set_index)]
            output.extend([
                str(row["total_agreement_candidate_1"]),
                str(row["total_agreement_candidate_2"]),
                format_optional(row["candidate_1_raw_logit"]),
                format_optional(row["candidate_2_raw_logit"]),
                format_optional(row["candidate_1_minus_candidate_2_raw_logit"]),
                str(bool(row["pair_has_answer_boundary_logits"])),
            ])
        matrix.append(output)
    return headers, matrix


truth_tables_by_model = {}
for model_id, all_rows in balanced_results_by_model.items():
    truth_tables_by_model[model_id] = {}
    output_dir = EXPERIMENT_ROOT / model_key(model_id)
    for reliability_exact in R_EXACT_VALUES:
        truth_tables_by_model[model_id][reliability_exact] = {}
        reliability_slug = R_LABEL_BY_EXACT[reliability_exact].replace("/", "_of_")
        for reasoning in REASONING_VALUES:
            headers, matrix = raw_truth_table(rows_at_condition(
                all_rows, reasoning=reasoning, reliability_exact=reliability_exact
            ))
            truth_tables_by_model[model_id][reliability_exact][reasoning] = {
                "headers": headers, "rows": matrix
            }
            path = output_dir / (
                f"agreement_truth_table_reliability_{reliability_slug}_"
                f"reasoning_{reasoning_label(reasoning).lower()}.csv"
            )
            with path.open("w", newline="", encoding="utf-8") as handle:
                writer = csv.writer(handle, lineterminator="\n")
                writer.writerow(headers)
                writer.writerows(matrix)


def display_truth_table(reasoning: bool, reliability_exact: str) -> None:
    for model_id, tables in truth_tables_by_model.items():
        table = tables[reliability_exact][reasoning]
        print(
            f"{model_id} — reliability {R_LABEL_BY_EXACT[reliability_exact]} — "
            f"reasoning {reasoning_label(reasoning)}: {len(table['rows'])} rows"
        )
        display(table["rows"][:2])


truth_widget = reasoning_selector()
truth_reliability_widget = reliability_selector()
truth_output = widgets.interactive_output(
    display_truth_table,
    {"reasoning": truth_widget, "reliability_exact": truth_reliability_widget},
)
display(widgets.VBox([widgets.HBox([truth_widget, truth_reliability_widget]), truth_output]))


## Compact completion audit

The audit view exposes the original completion, generated IDs, boundary metadata, and compliance outcome for one row without altering malformed text.


In [14]:
def display_audit_view(reasoning: bool) -> None:
    for model_id, rows in results_by_model.items():
        selected = rows_at_reasoning(rows, reasoning)
        sample = next(
            (row for row in selected if not row["strict_answer_compliance"]),
            selected[0],
        )
        print(f"{model_id} — reasoning {reasoning_label(reasoning)}")
        display({
            "row_id": sample["row_id"],
            "reliability": R_LABEL_BY_EXACT[row_reliability_exact(sample)],
            "presentation_order": sample["presentation_order"],
            "membership_sets": sample["membership_sets"],
            "observed_reports": sample["observed_reports"],
            "agreement_c1": sample["agreement_candidate_1_by_question"],
            "agreement_c2": sample["agreement_candidate_2_by_question"],
            "full_completion": sample["full_completion"],
            "generated_token_ids": sample["generated_token_ids"],
            "generated_sequence_token_ids": sample["generated_sequence_token_ids"],
            "reasoning_length_tokens": sample["reasoning_length_tokens"],
            "answer_boundary_source": sample["answer_boundary_source"],
            "answer_boundary_generated_token_count": sample["answer_boundary_generated_token_count"],
            "answer_leading_whitespace": repr(sample["answer_leading_whitespace"]),
            "answer_trailing_whitespace": repr(sample["answer_trailing_whitespace"]),
            "answer_leading_whitespace_token_ids": sample["answer_leading_whitespace_token_ids"],
            "answer_value_generated_token_ids": sample["answer_value_generated_token_ids"],
            "answer_trailing_whitespace_token_ids": sample["answer_trailing_whitespace_token_ids"],
            "terminal_stop_token_id": sample["completion"].get("terminal_stop_token_id"),
            "candidate_1_answer_token_ids": sample["candidate_1_answer_token_ids"],
            "candidate_2_answer_token_ids": sample["candidate_2_answer_token_ids"],
            "semantic_answer_compliance": sample["semantic_answer_compliance"],
            "exact_answer_format_compliance": sample["exact_answer_format_compliance"],
            "strict_answer_compliance": sample["strict_answer_compliance"],
            "compliance_break": sample["compliance_break"],
            "answer_boundary_break": sample["answer_boundary_break"],
            "candidate_1_raw_logit": sample["candidate_1_raw_logit"],
            "candidate_2_raw_logit": sample["candidate_2_raw_logit"],
            "activation_path": sample.get("activation_path"),
        })


audit_widget = reasoning_selector()
audit_output = widgets.interactive_output(display_audit_view, {"reasoning": audit_widget})
display(widgets.VBox([audit_widget, audit_output]))
